In [1]:
import datetime
import json
import pickle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis
import onnxruntime as ort
import matplotlib.pyplot as plt

BASE_DIR = Path("/app")
TESTSET_PATH = BASE_DIR / "testsets/four-people_testset.json"
OUTPUT_ROOT = BASE_DIR / "notebooks/data"
FACE_CROPS_DIR = OUTPUT_ROOT / "extracted_faces"
FACE_CROPS_DIR.mkdir(parents=True, exist_ok=True)

DET_SIZE = (640, 640)

print(f"Testset: {TESTSET_PATH}")
print(f"Face crops will be saved to: {FACE_CROPS_DIR}")

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Testset: /app/testsets/four-people_testset.json
Face crops will be saved to: /app/notebooks/data/extracted_faces


In [2]:
def init_insightface():
    """Initialize InsightFace with both detection AND recognition modules"""
    available = ort.get_available_providers()
    prefer_cuda = "CUDAExecutionProvider" in available
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if prefer_cuda else ["CPUExecutionProvider"]
    ctx_id = 0 if prefer_cuda else -1
    try:
        # Enable both detection and recognition for full face identification
        app = FaceAnalysis(allowed_modules=["detection", "recognition"], providers=providers)
        app.prepare(ctx_id=ctx_id, det_size=DET_SIZE)
        print(f"InsightFace initialized with providers={providers}, ctx_id={ctx_id}")
        return app
    except Exception as e:
        if prefer_cuda:
            print(f"GPU init failed ({e}); retrying on CPU only.")
            providers = ["CPUExecutionProvider"]
            ctx_id = -1
            app = FaceAnalysis(allowed_modules=["detection", "recognition"], providers=providers)
            app.prepare(ctx_id=ctx_id, det_size=DET_SIZE)
            print(f"InsightFace initialized with providers={providers}, ctx_id={ctx_id}")
            return app
        raise

app = init_insightface()


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvid

In [3]:
# Load testset
with open(TESTSET_PATH, "r", encoding="utf-8") as f:
    test_items = json.load(f)

print(f"Loaded {len(test_items)} images from test set")


Loaded 2503 images from test set


In [4]:
# Extract faces and embeddings from all test images
face_data_list = []
face_id_counter = 0
images_with_no_faces = 0

print("Extracting faces and embeddings from test images...")

for item in tqdm(test_items, desc="Processing images"):
    img_path = BASE_DIR / item.get("path", "")
    labels = item.get("labels", [])  # Get labels from testset
    
    if not img_path.exists():
        print(f"⚠️  Image not found: {img_path}")
        # Add entry with null embedding and face_image_path
        face_data_list.append({
            "image_path": str(img_path),
            "face_image_path": None,
            "embedding": None,
            "labels": ", ".join(labels) if labels else ""
        })
        continue
    
    # Load image
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"⚠️  Unable to load image: {img_path}")
        # Add entry with null embedding and face_image_path
        face_data_list.append({
            "image_path": str(img_path),
            "face_image_path": None,
            "embedding": None,
            "labels": ", ".join(labels) if labels else ""
        })
        continue
    
    # Detect faces
    faces = app.get(img)
    
    if not faces:
        print(f"⚠️  No faces detected in: {img_path}")
        images_with_no_faces += 1
        # Add entry with null embedding and face_image_path
        face_data_list.append({
            "image_path": str(img_path),
            "face_image_path": None,
            "embedding": None,
            "labels": ", ".join(labels) if labels else ""
        })
        continue
    
    # Get image dimensions
    img_height, img_width = img.shape[:2]
    
    # Process each detected face
    for face_idx, face in enumerate(faces):
        face_id_counter += 1
        
        # Extract face bounding box and clip to image bounds
        x1, y1, x2, y2 = [int(v) for v in face.bbox]
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_width, x2)
        y2 = min(img_height, y2)
        
        # Validate bounding box after clipping
        if x1 >= x2 or y1 >= y2:
            print(f"⚠️  Invalid bbox after clipping for: {img_path}")
            print(f"   Original bbox: {face.bbox}")
            print(f"   Clipped bbox: ({x1}, {y1}, {x2}, {y2})")
            print(f"   Image dimensions: {img_width}x{img_height}")
            continue
        
        # Extract face crop
        face_crop = img[y1:y2, x1:x2]
        
        # Check if face crop is valid
        if face_crop.size == 0:
            print(f"⚠️  Empty face crop for: {img_path}")
            print(f"   Bounding box: ({x1}, {y1}, {x2}, {y2})")
            print(f"   Image shape: {img.shape}")
            continue
        
        # Generate unique filename for face crop
        face_filename = f"face_{face_id_counter:06d}_{img_path.stem}_f{face_idx}.jpg"
        face_crop_path = FACE_CROPS_DIR / face_filename
        
        # Save face crop
        try:
            success = cv2.imwrite(str(face_crop_path), face_crop)
            if not success:
                raise Exception("cv2.imwrite returned False")
        except Exception as e:
            print(f"⚠️  Failed to write face crop for: {img_path}")
            print(f"   Error: {e}")
            print(f"   Face crop shape: {face_crop.shape}")
            print(f"   Bounding box: ({x1}, {y1}, {x2}, {y2})")
            
            # Display original image and face crop for debugging
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            
            # Original image with bounding box
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[0].imshow(img_rgb)
            axes[0].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                           fill=False, edgecolor='red', linewidth=2))
            axes[0].set_title(f"Original Image\n{img_path.name}")
            axes[0].axis('off')
            
            # Face crop
            face_crop_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            axes[1].imshow(face_crop_rgb)
            axes[1].set_title(f"Face Crop (failed to save)\n{face_filename}\nShape: {face_crop.shape}")
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
            continue
        
        # Get embedding
        embedding = face.normed_embedding
        
        # Store data
        face_data_list.append({
            "image_path": str(img_path),
            "face_image_path": str(face_crop_path),
            "embedding": embedding.tolist(),  # Convert numpy array to list
            "labels": ", ".join(labels) if labels else ""  # Join labels as comma-separated string
        })

print(f"\n✓ Extracted {len(face_data_list)} entries")
print(f"✓ Images with no faces detected: {images_with_no_faces}")
print(f"✓ Face crops saved to: {FACE_CROPS_DIR}")


Extracting faces and embeddings from test images...


Processing images:   0%|          | 0/2503 [00:00<?, ?it/s]/opt/venv/lib/python3.12/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
Processing images:   1%|          | 23/2503 [00:02<02:19, 17.73it/s]

⚠️  No faces detected in: /app/Images/four-people Testset/Donald Trump/01-00353749000007x.jpg


Processing images:   2%|▏         | 41/2503 [00:04<03:33, 11.53it/s]

⚠️  No faces detected in: /app/Images/four-people Testset/Donald Trump/01-00353749000031x.jpg


...

⚠️  No faces detected in: /app/Images/four-people Testset/None/LP_23555970.jpg


Processing images: 100%|██████████| 2503/2503 [03:05<00:00, 15.11it/s]

⚠️  No faces detected in: /app/Images/four-people Testset/None/LP_23698584.jpg
⚠️  No faces detected in: /app/Images/four-people Testset/None/LP_23806135.jpg


Processing images: 100%|██████████| 2503/2503 [03:05<00:00, 13.47it/s]


✓ Extracted 10120 entries
✓ Images with no faces detected: 171
✓ Face crops saved to: /app/notebooks/data/extracted_faces


In [5]:
# Save to JSON
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
json_output_path = OUTPUT_ROOT / f"face_embeddings_{timestamp}.json"

with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(face_data_list, f, indent=2)

print(f"✓ Saved face data to JSON: {json_output_path}")


✓ Saved face data to JSON: /app/notebooks/data/face_embeddings_20260131_153514.json


In [6]:
# Create DataFrame and save to CSV
df = pd.DataFrame(face_data_list)

csv_output_path = OUTPUT_ROOT / f"face_embeddings_{timestamp}.csv"
df.to_csv(csv_output_path, index=False)

print(f"✓ Saved face data to CSV: {csv_output_path}")
print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())


✓ Saved face data to CSV: /app/notebooks/data/face_embeddings_20260131_153514.csv

DataFrame shape: (10120, 4)
Columns: ['image_path', 'face_image_path', 'embedding', 'labels']

First few rows:
                                          image_path  \
0  /app/Images/four-people Testset/Donald Trump/0...   
1  /app/Images/four-people Testset/Donald Trump/0...   
2  /app/Images/four-people Testset/Donald Trump/0...   
3  /app/Images/four-people Testset/Donald Trump/0...   
4  /app/Images/four-people Testset/Donald Trump/0...   

                                     face_image_path  \
0  /app/notebooks/data/extracted_faces/face_00000...   
1  /app/notebooks/data/extracted_faces/face_00000...   
2  /app/notebooks/data/extracted_faces/face_00000...   
3  /app/notebooks/data/extracted_faces/face_00000...   
4  /app/notebooks/data/extracted_faces/face_00000...   

                                           embedding        labels  
0  [-0.048733823001384735, 0.07168117165565491, -...  Donald Tr

In [7]:
# Display summary statistics
print("\n" + "=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)
print(f"Total test images processed: {len(test_items)}")
print(f"Total faces extracted: {len(face_data_list)}")
print(f"Average faces per image: {len(face_data_list) / max(len(test_items), 1):.2f}")
print(f"\nOutput files generated:")
print(f"  • {json_output_path.name}")
print(f"  • {csv_output_path.name}")
print(f"  • {len(face_data_list)} face crops in {FACE_CROPS_DIR.name}/")
print("=" * 70)



EXTRACTION SUMMARY
Total test images processed: 2503
Total faces extracted: 10120
Average faces per image: 4.04

Output files generated:
  • face_embeddings_20260131_153514.json
  • face_embeddings_20260131_153514.csv
  • 10120 face crops in extracted_faces/
